In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_C.xpt"
csv_file = DATA_DIR / "SMQFAM_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 SMQFAM_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD410",
    "SMD415",
    "SMD415A",
    "SMD415B",
    "SMD415C",
    "SMD430",
    "SMD440",
    "SMD450"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQFAM_C column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10122:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 9:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,122 rows x 9 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)

true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Tiny-value artifacts were unexpectedly "
        "found in SMQFAM_C. Do not automatically replace."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifact exists."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT:
#
# Never round real measurements.
#
# Even though this particular SMQFAM file has no
# genuine fractional values, check first.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in SMQFAM_C."
)


# ============================================================
# 6. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 21005
    or
    df["SEQN"].max() != 31126
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# All eight SMQFAM variables are whole-number
# codes/counts in this released file.
#
# No .round() is used.
#

DATA_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in DATA_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Questionnaire/count fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC CHECK - SMD410
# ============================================================

EXPECTED_SMD410 = {
    1: 2096,
    2: 7884,
    7: 2,
    9: 7
}


actual = (
    df["SMD410"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- SMD410 ---")


if (
    actual != EXPECTED_SMD410
    or
    int(df["SMD410"].isna().sum()) != 133
):

    raise ValueError(
        "SMD410 differs from CDC."
    )


print(
    "PASS: SMD410 matches CDC."
)


# ============================================================
# 9. CDC CHECK - SMD415
# ============================================================

s = df["SMD415"]


print("\n--- SMD415 ---")

print(
    "1-2:",
    int(s.between(1, 2).sum())
)

print(
    "3 or more:",
    int(s.eq(3).sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)


if not (
    int(s.between(1, 2).sum()) == 1917
    and
    int(s.eq(3).sum()) == 179
    and
    int(s.isna().sum()) == 8026
):

    raise ValueError(
        "SMD415 differs from CDC."
    )


print(
    "PASS: SMD415 matches CDC."
)


# ============================================================
# 10. CDC CHECK - SMD415A
# ============================================================

s = df["SMD415A"]


if not (
    int(s.between(1, 2).sum()) == 1862
    and
    int(s.eq(3).sum()) == 173
    and
    int(s.isna().sum()) == 8087
):

    raise ValueError(
        "SMD415A differs from CDC."
    )


print(
    "PASS: SMD415A matches CDC."
)


# ============================================================
# 11. CDC CHECK - SMD415B
# ============================================================

EXPECTED_SMD415B = {
    1: 194,
    2: 19
}


actual = (
    df["SMD415B"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_SMD415B
    or
    int(df["SMD415B"].isna().sum()) != 9909
):

    raise ValueError(
        "SMD415B differs from CDC."
    )


print(
    "PASS: SMD415B matches CDC."
)


# ============================================================
# 12. CDC CHECK - SMD415C
# ============================================================

EXPECTED_SMD415C = {
    1: 90,
    2: 10
}


actual = (
    df["SMD415C"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_SMD415C
    or
    int(df["SMD415C"].isna().sum()) != 10022
):

    raise ValueError(
        "SMD415C differs from CDC."
    )


print(
    "PASS: SMD415C matches CDC."
)


# ============================================================
# 13. CDC CHECK - SMD430
# ============================================================

#
# CDC released frequencies:
#
# 1-37 = 1,807
# 40   = 186
# Missing = 8,129
#
# Although questionnaire instructions allowed
# "IF NONE, ENTER 0", there are NO released
# zero records in this file.
#

s = df["SMD430"]


if not (
    int(s.between(1, 37).sum()) == 1807
    and
    int(s.eq(40).sum()) == 186
    and
    int(s.eq(0).sum()) == 0
    and
    int(s.isna().sum()) == 8129
):

    raise ValueError(
        "SMD430 differs from CDC."
    )


print(
    "PASS: SMD430 matches CDC."
)


# ============================================================
# 14. CDC CHECK - SMD440
# ============================================================

#
# CDC:
#
# 1-2 = 147
# 3   = 58
# Missing = 9,917
#

s = df["SMD440"]


if not (
    int(s.between(1, 2).sum()) == 147
    and
    int(s.eq(3).sum()) == 58
    and
    int(s.eq(0).sum()) == 0
    and
    int(s.isna().sum()) == 9917
):

    raise ValueError(
        "SMD440 differs from CDC."
    )


print(
    "PASS: SMD440 matches CDC."
)


# ============================================================
# 15. CDC CHECK - SMD450
# ============================================================

#
# CDC:
#
# 1-2 = 63
# 3   = 31
# Missing = 10,028
#

s = df["SMD450"]


if not (
    int(s.between(1, 2).sum()) == 63
    and
    int(s.eq(3).sum()) == 31
    and
    int(s.eq(0).sum()) == 0
    and
    int(s.isna().sum()) == 10028
):

    raise ValueError(
        "SMD450 differs from CDC."
    )


print(
    "PASS: SMD450 matches CDC."
)


# ============================================================
# 16. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# IMPORTANT:
#
# There is NO zero restoration in this file.
#
# Expected values are therefore exactly the original XPT.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    source_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(failures)
    )


print(
    "PASS: All 9 variables preserve "
    "the original XPT values exactly."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# No float_format.
# No rounding.
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 18. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 19. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 9 columns match source XPT values."
)


# ============================================================
# 20. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 21. RAW CSV FORMATTING
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("SMQFAM_C CONVERSION COMPLETE")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Tiny-value corrections: NONE")
print("Genuine decimal variables: NONE")
print("XPT -> CSV differences: 0 expected")
print("Missing-value differences: 0 expected")
print("Unnecessary .0: 0 expected")

NHANES 2003-2004 SMQFAM_C XPT -> CSV
Rows: 10,122
Columns: 9
PASS: Structure = 10,122 rows x 9 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifact exists.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in SMQFAM_C.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126
PASS: SEQN validated.

PASS: Questionnaire/count fields formatted without unnecessary .0.

--- SMD410 ---
PASS: SMD410 matches CDC.

--- SMD415 ---
1-2: 1917
3 or more: 179
Missing: 8026
PASS: SMD415 matches CDC.
PASS: SMD415A matches CDC.
PASS: SMD415B matches CDC.
PASS: SMD415C matches CDC.
PASS: SMD430 matches CDC.
PASS: SMD440 matches CDC.
PASS: SMD450 matches CDC.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: All 9 variables preserve the original XPT values exactly.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\